# AimSegDL full training pipeline

### Data preparation

In [ ]:
from aimsegdl.data_preparation.data_preparation import split_dataset

In [ ]:
split_dataset(
    in_root = "datasets_em",
    out_root = "prepared_data_em",
    create_test_split = True,
)

### Model training

In [ ]:
import torch
from aimsegdl.training.config import TrainingConfig
from aimsegdl.training.train import train

In [ ]:
train_config = TrainingConfig(
    learning_rate = 1e-3,
    batch_size = 8,
    num_epochs = 2000,
    num_workers = 0,
    image_height = 512,
    image_width = 512,
    pin_memory = True,
    device = "cuda" if torch.cuda.is_available() else "cpu",
    fibre_threshold = 0.5,
    axon_threshold = 0.5,
    min_diameter = 30.0,
    train_dir = "prepared_data_em/train_tiles/",
    val_dir = "prepared_data_em/val_tiles/",
    pretrained_weights = None,     # Path to .pth weights file or model name (if available) to initialize model
    load_checkpoint = False,       # Resume from checkpoint
    model_name = "aimsegdl",
    )

In [ ]:
%matplotlib inline
train(train_config)

### Model evaluation

In [ ]:
import torch
from aimsegdl.evaluation.evaluate_model import evaluate_model_on_testset, compute_imagewise_means, plot_summary_bar

In [ ]:
test_dir = "prepared_data_em/test_tiles/"
model_path = "best_weights_model.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

%matplotlib inline
fibre_results, axon_results = evaluate_model_on_testset(
    model_path,
    test_dir,
    device,
    output_csv = "Evaluation_Results",
    display_figure=False,
    show_plots = True
)

In [ ]:
#Plot overall performance metrics
mean_metrics = compute_imagewise_means(fibre_results)
plot_summary_bar(mean_metrics, "Fibre")

mean_metrics = compute_imagewise_means(axon_results)
plot_summary_bar(mean_metrics, "Axon")

### Export bioimageio model

In [ ]:
from aimsegdl.export_utils.create_bioimageio_model import export_bioimageio
from aimsegdl.export_utils.config import BioimageioExportConfig

In [ ]:
bioimageio_config = BioimageioExportConfig(
    model_path = f"best_weights_model.pth",
    model_name = "aimsegdl",
    model_version = "0.1.0",
    test_img_path = r"aimsegdl\example_images\P03A_Frame5_roi0.tif",
    output_dir = "bioimageio_model",
    model_pixel_size = 0.008,
    citation_text = "Carrillo-Barberà et al., 2025",
    citation_doi = "10.1234/fake-doi-placeholder",
    author_names = ["Pau Carrillo-Barberà", "Thibaut Goldsborough", "Alan O'Callaghan", "Andrea Poveda Sabuco", "Chiara Sgattoni", "Jose Antonio Gómez Sánchez", "Ana Rondelli", "Anna Williams", "Peter Bankhead"],
    license_id = "CC-BY-4.0",
)

In [ ]:
export_bioimageio(bioimageio_config)